# 04 · Model router – by prompt complexity
Two switchable modes (`complexity-mode`):
* **heuristic** – policy scores code, reasoning verbs, multi-questions, math, conversation depth → tier. Free and fast.
* **llm** – a small classifier model labels the prompt `simple|medium|complex`; falls back to heuristic if it fails.

`router-mode = hybrid` takes the *higher* tier of length and complexity.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

PROMPTS = {
  'trivia':   "What colour is the Space Stone?",
  'summary':  "Summarise the main arc of the Infinity Saga and compare Thor's and Iron Man's character development.",
  'complex':  "Design a step by step architecture for a multi-region event-driven platform. Analyze trade-offs, evaluate failure modes, "
              "and explain why you'd pick each component. Include a Python class for the retry logic. What are the pros and cons? How would you test it?",
}
USER = 'tony-stark'

set_router(router_mode='complexity', complexity_mode='heuristic', complexity_threshold='3', wait_s=60)
RESULTS = ask_all(PROMPTS, user=USER)
show(RESULTS, cols=('tier', 'model', 'reason'))

**Check the gateway log** – the log has no prompt text, so rows are matched to prompts by user + prompt token count. Takes a few minutes to appear:

In [ ]:
check_llm_log(RESULTS, user=USER)   # which deployment actually served each prompt, according to the gateway log

Switch to the **LLM classifier** and compare decisions:

In [ ]:
set_router(complexity_mode='llm', wait_s=60)
RESULTS = ask_all(PROMPTS, user=USER)
show(RESULTS, cols=('tier', 'model', 'reason'))

**Check the gateway log** (LLM gateway log, a few minutes of lag): was the classifier's tier really the model that served it? `llmTier` in the `reason` above is the classifier's verdict (0 simple, 1 medium, 2 complex); `-1` means the classifier call failed and the heuristic was used instead.

In [ ]:
check_llm_log(RESULTS, user=USER)   # which deployment actually served each prompt, according to the gateway log

Back to the default **hybrid** (length + complexity, heuristic):

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

set_router(router_mode='hybrid', complexity_mode='heuristic', length_threshold_tokens='500')